# TerraSpectra — Day 7: 3D Hyperspectral Tensor Preparation with PyTorch

**Project:** TerraSpectra — Hyperspectral Crop Disease Forecasting  
**Day:** 7 — 19 November 2026  
**ML task:** Convert annotated hyperspectral pixel locations into spatial-spectral 3D patches and prepare PyTorch `Dataset`, `DataLoader`, and tensor batches.

## Scientific scope and limitations

- The annotation CSV provides pixel coordinates, not verified healthy/diseased labels.
- This notebook creates **unlabelled spatial-spectral input patches** for a future 3D-CNN pipeline.
- It does not train a classifier, assign disease labels, or report disease-detection accuracy.
- Patch shape is configurable. The default is 9 × 9 spatial pixels across all 204 spectral bands.
- PyTorch `Conv3d` input convention is `(batch, channels, depth, height, width)`. Here, depth is the spectral dimension.

## Inputs

Upload:
1. `hyperspectral_day04_preprocessed.npy` (preferred), **or** `REFLECTANCE_2047.hdr` and `REFLECTANCE_2047.dat`.
2. `pos_Plant21.csv`.
3. Optional Day 6 split coordinate files: `train_coordinates.csv`, `validation_coordinates.csv`, `test_coordinates.csv`.

If Day 6 coordinate files are not supplied, the notebook recreates the same deterministic 70:15:15 split using `random_state=42`.

## Outputs

Creates a `TerraSpectra_Day07/` directory containing tensor batches, coordinate metadata, a tensor-pipeline report, and a ZIP archive.

## 1. Install and import dependencies

In [ ]:
!pip -q install spectral scikit-learn

import os
import json
import zipfile
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
PATCH_SIZE = 9
BATCH_SIZE = 16
EXPECTED_SHAPE = (512, 512, 204)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", device)
print("CUDA available:", torch.cuda.is_available())

## 2. Upload the hyperspectral cube and coordinate files

In [ ]:
from google.colab import files

uploaded = files.upload()

print("\nUploaded files:")
for filename in uploaded:
    print(" -", filename)

print("\nNPY files:", [f for f in os.listdir("/content") if f.endswith(".npy")])

## 3. Load the Day 4 cube, or reproduce Day 2 normalization

In [ ]:
day4_path = "/content/hyperspectral_day04_preprocessed.npy"
day2_path = "/content/processed_data/normalized_hyperspectral_data.npy"

if os.path.exists(day4_path):
    cube = np.load(day4_path).astype(np.float32)
    cube_source = "Day 4 preprocessed cube"
elif os.path.exists(day2_path):
    cube = np.load(day2_path).astype(np.float32)
    cube_source = "Day 2 per-band normalized cube"
else:
    hdr_path = "/content/REFLECTANCE_2047.hdr"
    dat_path = "/content/REFLECTANCE_2047.dat"
    if not (os.path.exists(hdr_path) and os.path.exists(dat_path)):
        raise FileNotFoundError(
            "Upload hyperspectral_day04_preprocessed.npy, "
            "processed_data/normalized_hyperspectral_data.npy, "
            "or both REFLECTANCE_2047.hdr and REFLECTANCE_2047.dat."
        )

    import spectral
    envi_image = spectral.open_image(hdr_path)
    original_cube = np.asarray(envi_image.load(), dtype=np.float32)

    # Match Day 2: per-band Z-score normalization across spatial axes.
    band_mean = original_cube.mean(axis=(0, 1), keepdims=True)
    band_std = original_cube.std(axis=(0, 1), keepdims=True)
    band_std[band_std == 0] = 1.0
    cube = ((original_cube - band_mean) / band_std).astype(np.float32)
    cube_source = "Original ENVI cube with Day 2 per-band Z-score normalization"

print("Cube source:", cube_source)
print("Cube shape:", cube.shape)
print("Cube dtype:", cube.dtype)
print("All values finite:", bool(np.isfinite(cube).all()))

assert cube.shape == EXPECTED_SHAPE, f"Expected {EXPECTED_SHAPE}, got {cube.shape}"
assert np.isfinite(cube).all(), "Cube contains NaN or infinite values."

## 4. Load and validate annotated coordinates

In [ ]:
csv_path = "/content/pos_Plant21.csv"
if not os.path.exists(csv_path):
    raise FileNotFoundError("Upload pos_Plant21.csv.")

coordinates = pd.read_csv(csv_path, header=None, names=["row", "col"])
coordinates["row"] = pd.to_numeric(coordinates["row"], errors="raise").astype(int)
coordinates["col"] = pd.to_numeric(coordinates["col"], errors="raise").astype(int)

input_count = len(coordinates)
coordinates = coordinates.drop_duplicates(["row", "col"]).reset_index(drop=True)

valid = (
    coordinates["row"].between(0, cube.shape[0] - 1)
    & coordinates["col"].between(0, cube.shape[1] - 1)
)
if not valid.all():
    raise ValueError("One or more annotation coordinates fall outside the cube.")

print("CSV rows:", input_count)
print("Unique valid coordinates:", len(coordinates))
display(coordinates.head())

## 5. Load the Day 6 split, or reproduce it

In [ ]:
split_files = {
    "train": "/content/train_coordinates.csv",
    "validation": "/content/validation_coordinates.csv",
    "test": "/content/test_coordinates.csv",
}

if all(os.path.exists(path) for path in split_files.values()):
    coord_train = pd.read_csv(split_files["train"])[["row", "col"]]
    coord_val = pd.read_csv(split_files["validation"])[["row", "col"]]
    coord_test = pd.read_csv(split_files["test"])[["row", "col"]]
    split_source = "Day 6 coordinate split files"
else:
    coord_train, coord_temp = train_test_split(
        coordinates, test_size=0.30, random_state=RANDOM_STATE, shuffle=True
    )
    coord_val, coord_test = train_test_split(
        coord_temp, test_size=0.50, random_state=RANDOM_STATE, shuffle=True
    )
    split_source = "Recreated Day 6 70:15:15 split (random_state=42)"

coord_train = coord_train.reset_index(drop=True)
coord_val = coord_val.reset_index(drop=True)
coord_test = coord_test.reset_index(drop=True)

print("Split source:", split_source)
print("Train:", len(coord_train))
print("Validation:", len(coord_val))
print("Test:", len(coord_test))

## 6. Define spatial-spectral patch extraction

In [ ]:
if PATCH_SIZE < 1 or PATCH_SIZE % 2 == 0:
    raise ValueError("PATCH_SIZE must be a positive odd integer.")

def extract_patches(hyperspectral_cube, coord_df, patch_size=9):
    \"\"\"Extract centered H x W x spectral-band patches with edge padding.\"\"\"
    radius = patch_size // 2
    padded = np.pad(
        hyperspectral_cube,
        ((radius, radius), (radius, radius), (0, 0)),
        mode="reflect"
    )

    patches = []
    for row, col in coord_df[["row", "col"]].itertuples(index=False, name=None):
        patch = padded[
            row:row + patch_size,
            col:col + patch_size,
            :
        ]
        patches.append(patch)

    if not patches:
        return np.empty((0, patch_size, patch_size, hyperspectral_cube.shape[-1]),
                        dtype=np.float32)

    return np.stack(patches).astype(np.float32)

train_patches = extract_patches(cube, coord_train, PATCH_SIZE)
val_patches = extract_patches(cube, coord_val, PATCH_SIZE)
test_patches = extract_patches(cube, coord_test, PATCH_SIZE)

print("Train patches:", train_patches.shape)
print("Validation patches:", val_patches.shape)
print("Test patches:", test_patches.shape)

## 7. Convert patches to PyTorch 3D tensor format

In [ ]:
def to_conv3d_layout(patches):
    # Input patch layout: (N, H, W, spectral_bands)
    # Conv3d layout:      (N, channels, spectral_depth, H, W)
    tensor = torch.from_numpy(patches)
    tensor = tensor.permute(0, 3, 1, 2).unsqueeze(1).contiguous()
    return tensor.float()

X_train_tensor = to_conv3d_layout(train_patches)
X_val_tensor = to_conv3d_layout(val_patches)
X_test_tensor = to_conv3d_layout(test_patches)

print("PyTorch Conv3D tensor shapes:")
print("Train:", tuple(X_train_tensor.shape))
print("Validation:", tuple(X_val_tensor.shape))
print("Test:", tuple(X_test_tensor.shape))

expected_shape = (1, cube.shape[2], PATCH_SIZE, PATCH_SIZE)
assert tuple(X_train_tensor.shape[1:]) == expected_shape
assert tuple(X_val_tensor.shape[1:]) == expected_shape
assert tuple(X_test_tensor.shape[1:]) == expected_shape

## 8. Create a Dataset that retains pixel coordinates

In [ ]:
class HyperspectralPatchDataset(Dataset):
    def __init__(self, patch_tensor, coord_df):
        if len(patch_tensor) != len(coord_df):
            raise ValueError("Patch count and coordinate count must match.")
        self.patches = patch_tensor
        self.coordinates = coord_df[["row", "col"]].to_numpy(dtype=np.int64)

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, index):
        patch = self.patches[index]
        coordinate = torch.tensor(self.coordinates[index], dtype=torch.long)
        return patch, coordinate

train_dataset = HyperspectralPatchDataset(X_train_tensor, coord_train)
val_dataset = HyperspectralPatchDataset(X_val_tensor, coord_val)
test_dataset = HyperspectralPatchDataset(X_test_tensor, coord_test)

print("Dataset sizes:", len(train_dataset), len(val_dataset), len(test_dataset))

## 9. Build PyTorch DataLoaders

In [ ]:
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

print("Number of train batches:", len(train_loader))
print("Number of validation batches:", len(val_loader))
print("Number of test batches:", len(test_loader))

# Inspect one batch
sample_batch, sample_coordinates = next(iter(train_loader))
print("\nOne training batch:")
print("Tensor shape:", tuple(sample_batch.shape))
print("Coordinate batch shape:", tuple(sample_coordinates.shape))
print("Tensor dtype:", sample_batch.dtype)
print("All values finite:", bool(torch.isfinite(sample_batch).all()))

## 10. Verify tensor pipeline integrity

In [ ]:
for split_name, loader in [
    ("train", train_loader),
    ("validation", val_loader),
    ("test", test_loader),
]:
    total = 0
    for batch_x, batch_coords in loader:
        assert batch_x.ndim == 5
        assert batch_x.shape[1:] == (1, cube.shape[2], PATCH_SIZE, PATCH_SIZE)
        assert batch_coords.ndim == 2 and batch_coords.shape[1] == 2
        assert torch.isfinite(batch_x).all()
        total += batch_x.shape[0]
    print(f"{split_name}: verified {total} patches")

# Coordinate split overlap check
sets = [
    set(map(tuple, frame[["row", "col"]].to_numpy()))
    for frame in [coord_train, coord_val, coord_test]
]
assert sets[0].isdisjoint(sets[1])
assert sets[0].isdisjoint(sets[2])
assert sets[1].isdisjoint(sets[2])

print("All tensor and coordinate integrity checks passed.")

## 11. Save Day 7 tensor-pipeline artifacts

In [ ]:
output_dir = "/content/TerraSpectra_Day07"
os.makedirs(output_dir, exist_ok=True)

# Save CPU tensors for portability; load with map_location when needed.
torch.save(X_train_tensor.cpu(), os.path.join(output_dir, "X_train_patches.pt"))
torch.save(X_val_tensor.cpu(), os.path.join(output_dir, "X_validation_patches.pt"))
torch.save(X_test_tensor.cpu(), os.path.join(output_dir, "X_test_patches.pt"))

coord_train.to_csv(os.path.join(output_dir, "train_coordinates.csv"), index=False)
coord_val.to_csv(os.path.join(output_dir, "validation_coordinates.csv"), index=False)
coord_test.to_csv(os.path.join(output_dir, "test_coordinates.csv"), index=False)

report = pd.DataFrame([
    {"split": "train", "samples": len(train_dataset), "batches": len(train_loader)},
    {"split": "validation", "samples": len(val_dataset), "batches": len(val_loader)},
    {"split": "test", "samples": len(test_dataset), "batches": len(test_loader)},
])
report.to_csv(os.path.join(output_dir, "tensor_pipeline_report.csv"), index=False)

metadata = {
    "project": "TerraSpectra",
    "day": 7,
    "cube_source": cube_source,
    "split_source": split_source,
    "cube_shape": list(cube.shape),
    "spectral_bands": int(cube.shape[-1]),
    "patch_size": PATCH_SIZE,
    "tensor_layout": "(N, 1, spectral_bands, patch_height, patch_width)",
    "batch_size": BATCH_SIZE,
    "random_state": RANDOM_STATE,
    "labels_available": False,
    "purpose": "Unlabelled spatial-spectral input batches for future 3D-CNN work."
}
with open(os.path.join(output_dir, "tensor_pipeline_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

display(report)
print("Saved artifacts:", sorted(os.listdir(output_dir)))

## 12. Create a ZIP archive

In [ ]:
zip_path = "/content/TerraSpectra_Day07_Outputs.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for filename in os.listdir(output_dir):
        full_path = os.path.join(output_dir, filename)
        archive.write(full_path, arcname=os.path.join("TerraSpectra_Day07", filename))

print("Created:", zip_path)
from google.colab import files
files.download(zip_path)

## 13. Day 7 completion summary

This notebook:
- Extracts centered spatial-spectral patches around the annotated pixel coordinates.
- Converts patches to PyTorch 5D tensors suitable for `torch.nn.Conv3d`.
- Creates `Dataset` and `DataLoader` objects for train, validation, and test subsets.
- Retains the original row/column coordinate for every sample.
- Verifies tensor dimensions, finite values, and split separation.
- Saves tensor files and metadata for future model development.

**Important:** No class labels are supplied by the annotation CSV. These are unlabelled input tensors; do not train a supervised disease classifier or report disease accuracy from this notebook.

**Suggested Git commit:**

`Day 7: Add 3D hyperspectral tensor preparation and PyTorch data loaders`

**Repository location:**

`ml/notebooks/TerraSpectra_Day07_3D_Tensor_Batch_Pipeline.ipynb`